
# Seq2Seq 的數學推導

> 定義 Seq2Seq（vanilla RNN encoder–decoder）的 forward，再從 decoder output layer 開始做 BPTT（Backpropagation Through Time）完整推導。  
> 記號以向量形式表示，$\odot$ 表示逐元素相乘。  
> 本 notebook 對應的是 **encoder 與 decoder 都使用 vanilla RNN、decoder 初始 hidden 直接接 encoder 最後 hidden、training 使用 teacher forcing、loss 使用 average cross-entropy** 的版本。

Seq2Seq（Sequence-to-Sequence）是一種把一段序列轉換成另一段序列的模型。

可以把它想成：

* 輸入：一串東西（文字、數字、符號）
* 輸出：另一串東西（通常長度不一定一樣）

> 核心概念只有一句話：「先讀完整段輸入 → 再一個一個生成輸出」

<img src="image/Seq2seq_ani.gif" alt="seq2seq" style="width: 70%;"/>

---

## 0) 符號定義

給定：

- source sequence：
  $$
  \mathbf{x} = (\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_{T_x})
  $$
- target sequence：
  $$
  \mathbf{y}^{*} = (\mathbf{y}^{*}_1, \mathbf{y}^{*}_2, \dots, \mathbf{y}^{*}_{T_y})
  $$

其中：

- $\mathbf{x}_t \in \mathbb{R}^{V_x}$：encoder 在時間步 $t$ 的輸入（one-hot）
- $\mathbf{u}_t \in \mathbb{R}^{V_y}$：decoder 在時間步 $t$ 的輸入（teacher forcing 下通常為前一個正確 token 的 one-hot）
- $\mathbf{h}^{enc}_t \in \mathbb{R}^{H}$：encoder hidden state
- $\mathbf{h}^{dec}_t \in \mathbb{R}^{H}$：decoder hidden state
- $\mathbf{c} \in \mathbb{R}^{H}$：context vector
- $\mathbf{o}_t \in \mathbb{R}^{V_y}$：decoder 的 logits
- $\hat{\mathbf{y}}_t \in \mathbb{R}^{V_y}$：decoder 的 softmax 機率輸出

若採用 encoder / decoder 分開參數的寫法：

- encoder 參數：$(W_{xh}^{enc}, W_{hh}^{enc}, \mathbf{b}_h^{enc})$
- decoder 參數：$(W_{xh}^{dec}, W_{hh}^{dec}, \mathbf{b}_h^{dec})$
- output layer 參數：$(W_{ho}, \mathbf{b}_o)$

---

## 1) Seq2Seq Forward（向前傳播）



### 1.1 Encoder RNN

初始化：
$$
\mathbf{h}^{enc}_0 = \mathbf{0}
$$

在時間步 $t$，encoder 的 pre-activation 與 hidden state 為：
$$
\mathbf{a}^{enc}_t =
W_{xh}^{enc}\mathbf{x}_t + W_{hh}^{enc}\mathbf{h}^{enc}_{t-1} + \mathbf{b}_h^{enc}
$$

$$
\mathbf{h}^{enc}_t = \tanh(\mathbf{a}^{enc}_t)
$$

---

### 1.2 Context vector

encoder 讀完整個 source sequence 後，取最後 hidden state 作為 context vector：
$$
\mathbf{c} = \mathbf{h}^{enc}_{T_x}
$$

這也是 encoder 與 decoder 的橋接點。

---

### 1.3 Decoder RNN

decoder 初始 hidden state 設為：
$$
\mathbf{h}^{dec}_0 = \mathbf{c}
$$

若使用 teacher forcing，則 decoder 在時間步 $t$ 的輸入為：
$$
\mathbf{u}_t = \mathbf{y}^{*}_{t-1}
$$

其中 $t=1$ 時通常使用 `<sos>` 的 one-hot 向量。

decoder 的 pre-activation 與 hidden state：
$$
\mathbf{a}^{dec}_t =
W_{xh}^{dec}\mathbf{u}_t + W_{hh}^{dec}\mathbf{h}^{dec}_{t-1} + \mathbf{b}_h^{dec}
$$

$$
\mathbf{h}^{dec}_t = \tanh(\mathbf{a}^{dec}_t)
$$

---

### 1.4 Output layer

decoder 在每個時間步的 logits：
$$
\mathbf{o}_t = W_{ho}\mathbf{h}^{dec}_t + \mathbf{b}_o
$$

softmax 機率：
$$
\hat{\mathbf{y}}_t = \mathrm{softmax}(\mathbf{o}_t)
$$



## 2) Loss Function

若 target sequence 長度為 $T_y$，則每個時間步的 cross-entropy loss 為：
$$
\mathcal{L}_t = - \sum_{i=1}^{V_y} y^{*}_{t,i} \log \hat{y}_{t,i}
$$

整段 target sequence 的平均 loss 定義為：
$$
\mathcal{L} = \frac{1}{T_y}\sum_{t=1}^{T_y}\mathcal{L}_t
$$

這裡特別使用 **average loss**，因此 backward 時每個時間步的 logits 梯度也要除以 $T_y$。

---



## 3) 從 Output Layer 開始 Backward（Decoder 端）

### 3.1 Softmax + Cross Entropy 的梯度

定義：
$$
\delta_t^{o} \equiv \frac{\partial \mathcal{L}}{\partial \mathbf{o}_t}
$$

由 softmax 搭配 cross-entropy 的標準結果可得：
$$
\frac{\partial \mathcal{L}_t}{\partial \mathbf{o}_t}
=
\hat{\mathbf{y}}_t - \mathbf{y}^{*}_t
$$

但因為總 loss 是平均：
$$
\mathcal{L} = \frac{1}{T_y}\sum_{t=1}^{T_y}\mathcal{L}_t
$$

所以：
$$
\delta_t^{o}
=
\frac{\partial \mathcal{L}}{\partial \mathbf{o}_t}
=
\frac{1}{T_y}\left(\hat{\mathbf{y}}_t - \mathbf{y}^{*}_t\right)
$$

---

### 3.2 對 output layer 參數與 decoder hidden 的梯度

由
$$
\mathbf{o}_t = W_{ho}\mathbf{h}^{dec}_t + \mathbf{b}_o
$$

可得：

對 $W_{ho}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{ho}}
=
\sum_{t=1}^{T_y}\delta_t^{o}(\mathbf{h}^{dec}_t)^T
$$

對 $\mathbf{b}_o$：
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_o}
=
\sum_{t=1}^{T_y}\delta_t^{o}
$$

對 $\mathbf{h}^{dec}_t$（來自當前 output layer）：
$$
\left.\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{dec}_t}\right|_{\text{output}}
=
W_{ho}^T\delta_t^{o}
$$

---

### 3.3 Decoder hidden state 的總梯度

在 BPTT 中，$\mathbf{h}^{dec}_t$ 不只影響當前時間步輸出，也會透過 recurrent connection 影響未來時間步，因此定義：

$$
\delta_t^{h,dec} \equiv \frac{\partial \mathcal{L}}{\partial \mathbf{h}^{dec}_t}
$$

則有：
$$
\delta_t^{h,dec}
=
\underbrace{W_{ho}^T\delta_t^{o}}_{\text{當前輸出層}}
+
\underbrace{\delta_{t,\text{future}}^{h,dec}}_{\text{來自未來時間步}}
$$

在程式實作中，後面這項通常會寫成 `dh_next`。



## 4) Decoder RNN 的反向傳播（BPTT）

由 decoder hidden state 的定義：
$$
\mathbf{h}^{dec}_t = \tanh(\mathbf{a}^{dec}_t)
$$

其中
$$
\mathbf{a}^{dec}_t =
W_{xh}^{dec}\mathbf{u}_t + W_{hh}^{dec}\mathbf{h}^{dec}_{t-1} + \mathbf{b}_h^{dec}
$$

---

### 4.1 從 hidden 回傳到 pre-activation

因為
$$
\frac{d}{dz}\tanh(z)=1-\tanh^2(z)
$$

定義：
$$
\delta_t^{a,dec}
\equiv
\frac{\partial \mathcal{L}}{\partial \mathbf{a}^{dec}_t}
$$

則：
$$
\delta_t^{a,dec}
=
\delta_t^{h,dec}
\odot
\left(1-(\mathbf{h}^{dec}_t)^2\right)
$$

---

### 4.2 對 decoder 參數的梯度

對 $W_{xh}^{dec}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}\mathbf{u}_t^T
$$

對 $W_{hh}^{dec}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}(\mathbf{h}^{dec}_{t-1})^T
$$

對 $\mathbf{b}_h^{dec}$：
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}
$$

---

### 4.3 對前一個 decoder hidden 的梯度

由
$$
\mathbf{a}^{dec}_t =
W_{xh}^{dec}\mathbf{u}_t + W_{hh}^{dec}\mathbf{h}^{dec}_{t-1} + \mathbf{b}_h^{dec}
$$

可得：
$$
\delta_{t-1}^{h,dec}
=
(W_{hh}^{dec})^T \delta_t^{a,dec}
$$

這一項就是下一輪 backward 迴圈中的 future hidden gradient。



## 5) Encoder–Decoder 的梯度橋接（Seq2Seq 最關鍵）

這個 vanilla Seq2Seq 的橋接關係是：
$$
\mathbf{h}^{dec}_0 = \mathbf{c} = \mathbf{h}^{enc}_{T_x}
$$

因此，當 decoder 做完 BPTT 之後，最前面那一包 hidden gradient 會直接變成 encoder 最後 hidden state 的梯度：

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{enc}_{T_x}}
=
\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{dec}_0}
$$

若沿用前面的符號，可記為：
$$
\delta_{T_x}^{h,enc} = \delta_0^{h,dec}
$$

這條式子就是：

- decoder loss 為什麼能訓練 encoder
- 為什麼即使 encoder 自己沒有 output layer，仍然能收到梯度

的根本原因。

---



## 6) Encoder RNN 的反向傳播（BPTT）

encoder 的 hidden state：
$$
\mathbf{h}^{enc}_t = \tanh(\mathbf{a}^{enc}_t)
$$

其中
$$
\mathbf{a}^{enc}_t =
W_{xh}^{enc}\mathbf{x}_t + W_{hh}^{enc}\mathbf{h}^{enc}_{t-1} + \mathbf{b}_h^{enc}
$$

---

### 6.1 從最後 hidden 開始 backward

Encoder backward 的起點不是 output layer，而是 decoder 傳回來的 bridge gradient：
$$
\delta_{T_x}^{h,enc}
=
\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{enc}_{T_x}}
$$

---

### 6.2 從 hidden 回傳到 pre-activation

定義：
$$
\delta_t^{a,enc}
\equiv
\frac{\partial \mathcal{L}}{\partial \mathbf{a}^{enc}_t}
$$

則：
$$
\delta_t^{a,enc}
=
\delta_t^{h,enc}
\odot
\left(1-(\mathbf{h}^{enc}_t)^2\right)
$$

---

### 6.3 對 encoder 參數的梯度

對 $W_{xh}^{enc}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}\mathbf{x}_t^T
$$

對 $W_{hh}^{enc}$：
$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}(\mathbf{h}^{enc}_{t-1})^T
$$

對 $\mathbf{b}_h^{enc}$：
$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}
$$

---

### 6.4 對前一個 encoder hidden 的梯度

$$
\delta_{t-1}^{h,enc}
=
(W_{hh}^{enc})^T \delta_t^{a,enc}
$$

因此 encoder 也和一般 RNN 一樣，從最後一步一路反傳到最前面。



## 7) 把整個梯度流串起來看

若把整個 Seq2Seq 畫成梯度流，會是：

$$
\mathcal{L}
\rightarrow
\mathbf{o}_{T_y}, \mathbf{o}_{T_y-1}, \dots, \mathbf{o}_1
\rightarrow
\mathbf{h}^{dec}_{T_y}, \mathbf{h}^{dec}_{T_y-1}, \dots, \mathbf{h}^{dec}_0
=
\mathbf{h}^{enc}_{T_x}
\rightarrow
\mathbf{h}^{enc}_{T_x-1}, \dots, \mathbf{h}^{enc}_1
$$

也就是：

1. 先從 decoder 的每一步 logits 反傳
2. 經過 output layer 回到 decoder hidden states
3. decoder 做完整 BPTT
4. 最前面的 decoder hidden gradient 接到 encoder 最後 hidden
5. encoder 再做一遍 BPTT

所以這個模型雖然看起來像「兩個 RNN」，但在訓練時其實是一張完整連接的計算圖。

---



## 8) 最後整理：單一時間步的 backward 公式

### 8.1 Decoder output layer
$$
\delta_t^{o}
=
\frac{1}{T_y}\left(\hat{\mathbf{y}}_t - \mathbf{y}^{*}_t\right)
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{ho}}
=
\sum_{t=1}^{T_y}\delta_t^{o}(\mathbf{h}^{dec}_t)^T
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_o}
=
\sum_{t=1}^{T_y}\delta_t^{o}
$$

$$
\delta_t^{h,dec}
=
W_{ho}^T\delta_t^{o}
+
\delta_{t,\text{future}}^{h,dec}
$$

---

### 8.2 Decoder recurrent layer
$$
\delta_t^{a,dec}
=
\delta_t^{h,dec}\odot \left(1-(\mathbf{h}^{dec}_t)^2\right)
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}\mathbf{u}_t^T
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}(\mathbf{h}^{dec}_{t-1})^T
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{dec}}
=
\sum_{t=1}^{T_y}\delta_t^{a,dec}
$$

$$
\delta_{t-1}^{h,dec}
=
(W_{hh}^{dec})^T\delta_t^{a,dec}
$$

---

### 8.3 Bridge
$$
\delta_{T_x}^{h,enc} = \delta_0^{h,dec}
$$

---

### 8.4 Encoder recurrent layer
$$
\delta_t^{a,enc}
=
\delta_t^{h,enc}\odot \left(1-(\mathbf{h}^{enc}_t)^2\right)
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{xh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}\mathbf{x}_t^T
$$

$$
\frac{\partial \mathcal{L}}{\partial W_{hh}^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}(\mathbf{h}^{enc}_{t-1})^T
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_h^{enc}}
=
\sum_{t=1}^{T_x}\delta_t^{a,enc}
$$

$$
\delta_{t-1}^{h,enc}
=
(W_{hh}^{enc})^T\delta_t^{a,enc}
$$



## 9) 對照程式實作的對應關係

若對照 NumPy 版 Seq2Seq 實作，會有以下對應：

- `dlogits /= T_dec`
  - 對應：
  $$
  \delta_t^{o}
  =
  \frac{1}{T_y}\left(\hat{\mathbf{y}}_t-\mathbf{y}^{*}_t\right)
  $$

- `self.dWho += np.outer(dlogits, h)`
  - 對應：
  $$
  \frac{\partial \mathcal{L}}{\partial W_{ho}}
  +=
  \delta_t^{o}(\mathbf{h}_t^{dec})^T
  $$

- `dh = self.Who.T @ dlogits + dh_next`
  - 對應：
  $$
  \delta_t^{h,dec}
  =
  W_{ho}^T\delta_t^{o}
  +
  \delta_{t,\text{future}}^{h,dec}
  $$

- `da = (1.0 - h * h) * dh`
  - 對應：
  $$
  \delta_t^{a,dec}
  =
  \delta_t^{h,dec}\odot(1-(\mathbf{h}^{dec}_t)^2)
  $$

- `self.dWxh_dec += np.outer(da, x)`
  - 對應：
  $$
  \frac{\partial \mathcal{L}}{\partial W_{xh}^{dec}}
  +=
  \delta_t^{a,dec}\mathbf{u}_t^T
  $$

- `self.dWhh_dec += np.outer(da, h_prev)`
  - 對應：
  $$
  \frac{\partial \mathcal{L}}{\partial W_{hh}^{dec}}
  +=
  \delta_t^{a,dec}(\mathbf{h}^{dec}_{t-1})^T
  $$

- `dh_next = self.Whh_dec.T @ da`
  - 對應：
  $$
  \delta_{t-1}^{h,dec}
  =
  (W_{hh}^{dec})^T\delta_t^{a,dec}
  $$

- `d_enc_last = dh_next.copy()`
  - 對應：
  $$
  \delta_{T_x}^{h,enc}=\delta_0^{h,dec}
  $$

- `da = (1.0 - h * h) * dh_next`
  - 對應：
  $$
  \delta_t^{a,enc}
  =
  \delta_t^{h,enc}\odot(1-(\mathbf{h}^{enc}_t)^2)
  $$

- `dh_next = self.Whh_enc.T @ da`
  - 對應：
  $$
  \delta_{t-1}^{h,enc}
  =
  (W_{hh}^{enc})^T\delta_t^{a,enc}
  $$

---



## 10) 一句話總結

這份 Seq2Seq（vanilla RNN）在數學上的核心是：

$$
\text{Encoder RNN}
\rightarrow
\mathbf{c}
\rightarrow
\text{Decoder RNN}
\rightarrow
\mathcal{L}
$$

而 backward 的核心則是：

$$
\mathcal{L}
\rightarrow
\text{Decoder BPTT}
\rightarrow
\mathbf{h}^{dec}_0
=
\mathbf{h}^{enc}_{T_x}
\rightarrow
\text{Encoder BPTT}
$$

也就是說：

> **loss 雖然定義在 decoder 端，但梯度會穿過 context vector，回傳整個 encoder。**

這正是最經典 Seq2Seq 訓練機制的本質；而後來加入 attention，就是為了改善所有 source 資訊都被壓進單一 context vector 的瓶頸。
